# 이메일 발주서 첨부파일 추출

메일함(POP3)에서 제목에 "발주서/발주" 키워드가 있는 메일을 찾아 첨부파일만 다운로드합니다.

**진행 상태**
- 완료: 발주서 키워드로 메일 필터링 + 첨부파일 추출까지 (이 노트북)
- 이후 단계(추출된 첨부파일을 실제 발주서 데이터로 정리하는 작업)는 별도로 진행됨(다른 담당자)
- 향후 TODO: 첨부파일이 없고 본문에만 "발주" 관련 텍스트가 있는 메일도 캡처해서 뽑아내는 방법 검토 필요

원본: `파일 분류/메일 내용 불러오기/POP_mail.ipynb`(v1) → `POP_mail_v2.ipynb` → `POP_mail_v3.ipynb`. 세 버전 모두 이메일 계정/비밀번호가 평문으로 하드코딩되어 있어 그대로 옮기지 않았고, 가장 완성도 높은 v3(SSL 끊김 자동 재연결 + 체크포인트 재개) 로직만 가져와 자격증명을 `.env`에서 읽도록 정리했습니다.

실행 전 저장소 루트 `.env.example`을 참고해 `.env`에 **POP3_HOST / POP3_PORT / EMAIL_ADDR / EMAIL_PASSWORD** 값을 채워주세요. (EMAIL_PASSWORD는 메일 서비스에서 발급한 앱 비밀번호)

In [ ]:
import poplib
import email
from email.header import decode_header
from email import policy
import os, re, json, time
import pandas as pd

from dotenv import load_dotenv
load_dotenv()

poplib._MAXLINE = 1024 * 1024

In [ ]:
# ── 설정 (.env에서 읽어옴) ─────────────────────────────────
POP3_HOST    = os.environ["POP3_HOST"]
POP3_PORT    = int(os.environ.get("POP3_PORT", "995"))
USE_SSL      = True
EMAIL_USER   = os.environ["EMAIL_ADDR"]
EMAIL_PASS   = os.environ["EMAIL_PASSWORD"]

KEYWORD_LIST   = ["발주서", "발주", "발 주 서"]   # 제목에 하나라도 있으면 수집
EMAIL_EXCLUDE  = ["고려기프트", "조아기프트"]      # 제목/본문에 있으면 제외 (다른 경로로 이미 처리 중)
ATTACH_EXCLUDE_KW  = ["시안", "사업자", "등록증", "로고", "명세표", "통장", "은행"]  # 파일명에 이 단어 있으면 제외
ATTACH_EXCLUDE_EXT = [".ai", ".psd", ".eps"]       # 이 확장자는 무조건 제외

ATTACH_DIR      = "./attachments"
OUTPUT_EXCEL    = "./order_mail_list.xlsx"
CHECKPOINT_FILE = "./pop_checkpoint.json"

os.makedirs(ATTACH_DIR, exist_ok=True)
print(f"저장 경로: {os.path.abspath(ATTACH_DIR)}")

## 헬퍼 함수

In [ ]:
def decode_str(value):
    if not value:
        return ""
    result = []
    for part, charset in decode_header(value):
        if isinstance(part, bytes):
            try:
                result.append(part.decode(charset or "utf-8", errors="replace"))
            except LookupError:
                result.append(part.decode("utf-8", errors="replace"))
        else:
            result.append(part)
    return "".join(result)


def get_body(msg):
    body_text, body_html = "", ""
    if msg.is_multipart():
        for part in msg.walk():
            ct = part.get_content_type()
            cd = str(part.get("Content-Disposition", ""))
            if "attachment" in cd:
                continue
            charset = part.get_content_charset() or "utf-8"
            if ct == "text/plain" and not body_text:
                try:
                    body_text = part.get_payload(decode=True).decode(charset, errors="replace")
                except Exception:
                    pass
            elif ct == "text/html" and not body_html:
                try:
                    body_html = part.get_payload(decode=True).decode(charset, errors="replace")
                except Exception:
                    pass
    else:
        charset = msg.get_content_charset() or "utf-8"
        try:
            body_text = msg.get_payload(decode=True).decode(charset, errors="replace")
        except Exception:
            pass
    if body_text:
        return body_text.strip()
    return re.sub(r"<[^>]+>", "", body_html).strip() if body_html else ""


def has_keyword(text):
    return any(kw in text for kw in KEYWORD_LIST)


def is_excluded(text):
    return any(kw in text for kw in EMAIL_EXCLUDE)


def is_attach_excl(filename):
    """파일명에 제외 단어 포함 또는 제외 확장자면 True"""
    name = filename.lower()
    if any(kw in filename for kw in ATTACH_EXCLUDE_KW):
        return True
    if any(name.endswith(ext) for ext in ATTACH_EXCLUDE_EXT):
        return True
    return False


def reconnect():
    c = poplib.POP3_SSL(POP3_HOST, POP3_PORT) if USE_SSL else poplib.POP3(POP3_HOST, POP3_PORT)
    c.user(EMAIL_USER)
    c.pass_(EMAIL_PASS)
    return c


def save_checkpoint(i):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"next_i": i, "records": records}, f, ensure_ascii=False)


print("헬퍼 함수 정의 완료")

## 시작 방법 선택
`RESUME = True`면 체크포인트가 있을 때 이어서 처리하고, `False`면 처음부터 새로 시작합니다.

In [ ]:
RESUME = False

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        cp = json.load(f)
    saved_i = cp.get("next_i", 1)
    saved_n = len(cp.get("records", []))
    print(f"체크포인트 존재: {saved_i}번 메일부터 재개 가능 (기존 수집 {saved_n}개)")
    if not RESUME:
        os.remove(CHECKPOINT_FILE)
        print("RESUME=False → 체크포인트 삭제, 처음부터 시작")
else:
    print("체크포인트 없음 → 처음부터 시작")

## 메인 수집 루프
SSL 연결이 끊기면 자동 재연결하고, 진행 상황을 주기적으로 체크포인트 파일에 저장합니다.

In [ ]:
if RESUME and os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        cp = json.load(f)
    records     = cp.get("records", [])
    start_index = cp.get("next_i", 1)
    print(f"{start_index}번부터 재개 (기존 수집 {len(records)}개)")
else:
    records     = []
    start_index = 1
    print("처음부터 시작")

conn = reconnect()
num_messages = len(conn.list()[1])
print(f"받은메일함 총 {num_messages}개 / {start_index}~{num_messages}번 처리")

i = start_index
try:
    while i <= num_messages:
        try:
            header_bytes = b"\n".join(conn.top(i, 0)[1])
            header_msg   = email.message_from_bytes(header_bytes, policy=policy.compat32)
            subject      = decode_str(header_msg.get("Subject", ""))

            if is_excluded(subject) or not has_keyword(subject):
                i += 1
                continue

            if i % 200 == 0:
                print(f"  {i}/{num_messages} 처리 중... (수집 {len(records)}개)")
                save_checkpoint(i)

            raw_bytes = b"\n".join(conn.retr(i)[1])
            msg  = email.message_from_bytes(raw_bytes, policy=policy.compat32)
            body = get_body(msg)

            if is_excluded(body):
                i += 1
                continue

            sender = decode_str(msg.get("From", ""))
            date   = msg.get("Date", "")
            print(f"[{i}/{num_messages}] {subject}")

            saved = []
            if msg.is_multipart():
                for part in msg.walk():
                    if "attachment" not in str(part.get("Content-Disposition", "")):
                        continue
                    filename = decode_str(part.get_filename() or "")
                    if not filename or is_attach_excl(filename):
                        continue
                    base, ext = os.path.splitext(filename)
                    path = os.path.join(ATTACH_DIR, filename)
                    n = 1
                    while os.path.exists(path):
                        path = os.path.join(ATTACH_DIR, f"{base}_{n}{ext}")
                        n += 1
                    with open(path, "wb") as f:
                        f.write(part.get_payload(decode=True))
                    saved.append(os.path.abspath(path))
                    print(f"    [첨부] {os.path.basename(path)}")

            records.append({
                "수신일시":    date,
                "발신자":      sender,
                "제목":        subject,
                "본문":        body,
                "첨부파일경로": "\n".join(saved),
            })
            i += 1

        except KeyboardInterrupt:
            raise

        except Exception as e:
            print(f"\n[{i}] 연결 오류: {e} → 재연결 중...")
            save_checkpoint(i)
            time.sleep(5)
            try:
                conn.quit()
            except Exception:
                pass
            try:
                conn = reconnect()
                print(f"재연결 성공, {i}번부터 재개")
            except Exception as e2:
                print(f"재연결 실패: {e2} → 30초 후 재시도")
                time.sleep(30)
                conn = reconnect()

except KeyboardInterrupt:
    save_checkpoint(i)
    print(f"\n수동 중단 — {i}번에서 저장됨 (수집 {len(records)}개)")
    print("다음 실행 시 RESUME=True로 이어서 진행 가능")

else:
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)
    print(f"\n완료 - 총 {len(records)}개 메일 수집")

finally:
    try:
        conn.quit()
    except Exception:
        pass

## 엑셀로 저장

In [ ]:
if records:
    df = pd.DataFrame(records, columns=["수신일시", "발신자", "제목", "본문", "첨부파일경로"])
    with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="발주서메일")
        ws = writer.sheets["발주서메일"]
        for col, w in {"A": 28, "B": 35, "C": 45, "D": 80, "E": 60}.items():
            ws.column_dimensions[col].width = w
        from openpyxl.styles import Alignment
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
    print(f"저장 완료: {os.path.abspath(OUTPUT_EXCEL)}")
else:
    print("발주서 메일이 없습니다.")